# Phase 3 — Cart Abandonment Recovery

This notebook demonstrates the final enrichment pipeline.

The goal is to detect users who added an item to cart but did not purchase it in the same session.

Then the system checks whether the abandoned item's category is one of the user's top affinity categories.

Decision rule:

- If item category is in the user's top categories → `High_Discount`
- Otherwise → `Standard_Reminder`

In [2]:
import os
import sys

# Make sure PySpark uses the same Python environment as the notebook
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Hadoop Windows helper path
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"
os.environ["PATH"] = os.environ["PATH"] + r";C:\hadoop\bin"

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("Cart Abandonment Results Notebook")
    .master("local[1]")
    .config("spark.driver.memory", "2g")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.python.worker.reuse", "false")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

print("Spark started successfully")
print("Spark version:", spark.version)
print("Python used:", sys.executable)

Spark started successfully
Spark version: 3.5.8
Python used: c:\BigData_Project2\.venv\Scripts\python.exe


## Campaign Output

The campaign output was generated by `05_cart_abandonment.py`.

In [3]:
campaign_path = r"C:\BigData_Project2\output\campaign_output"

campaign_df = spark.read.parquet(campaign_path)

display(campaign_df.limit(30).toPandas())

,user_id,session_id,item_id,category,campaign_type
0,User_0,SESS_0085eddbbc,ITEM_1618,Electronics,Standard_Reminder
1,User_0,SESS_0085eddbbc,ITEM_4221,Electronics,Standard_Reminder
2,User_0,SESS_01b30b5c1f,ITEM_1911,Clothing,High_Discount
3,User_0,SESS_01b30b5c1f,ITEM_3530,Clothing,High_Discount
4,User_0,SESS_033691ac67,ITEM_1850,Clothing,High_Discount
5,User_0,SESS_0923a2ec2e,ITEM_1050,Books,High_Discount
6,User_0,SESS_0de6586001,ITEM_4436,Home,High_Discount
7,User_0,SESS_20cc19581c,ITEM_2583,Books,High_Discount
8,User_0,SESS_20ef7d6f73,ITEM_4754,Home,High_Discount
9,User_0,SESS_2904635e7a,ITEM_4115,Clothing,High_Discount


In [4]:
campaign_counts = campaign_df.groupBy("campaign_type").count()
display(campaign_counts.toPandas())

,campaign_type,count
0,Standard_Reminder,67842
1,High_Discount,132153


## Recommendation Query Demo

This section demonstrates the same logic as `06_query_demo.py`.

Example 1:

- User: `User_0`
- Item: `ITEM_1911`
- Product Category: Clothing
- User top categories include Clothing
- Result: `High_Discount`

Example 2:

- User: `User_0`
- Item: `ITEM_1618`
- Product Category: Electronics
- User top categories do not include Electronics
- Result: `Standard_Reminder`

In [5]:
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017"
DB_NAME = "ecommerce_recs"

client = MongoClient(MONGO_URI)
db = client[DB_NAME]


def recommend_campaign(user_id, item_id):
    user_profile = db.user_profiles.find_one({"user_id": user_id}, {"_id": 0})
    product = db.product_catalog.find_one({"item_id": item_id}, {"_id": 0})

    if not user_profile:
        return {"error": "User profile not found"}

    if not product:
        return {"error": "Product not found"}

    top_categories = user_profile.get("top_categories", [])
    top_category_names = [x["category"] for x in top_categories]

    item_category = product.get("category")

    campaign_type = (
        "High_Discount"
        if item_category in top_category_names
        else "Standard_Reminder"
    )

    return {
        "user_id": user_id,
        "item_id": item_id,
        "item_category": item_category,
        "item_brand": product.get("brand"),
        "user_top_categories": top_categories,
        "campaign_type": campaign_type
    }

In [6]:
recommend_campaign("User_0", "ITEM_1911")

{'user_id': 'User_0',
 'item_id': 'ITEM_1911',
 'item_category': 'Clothing',
 'item_brand': 'Zara',
 'user_top_categories': [{'category': 'Home', 'score': 150},
  {'category': 'Clothing', 'score': 124},
  {'category': 'Books', 'score': 113}],
 'campaign_type': 'High_Discount'}

In [7]:
recommend_campaign("User_0", "ITEM_1618")

{'user_id': 'User_0',
 'item_id': 'ITEM_1618',
 'item_category': 'Electronics',
 'item_brand': 'Dell',
 'user_top_categories': [{'category': 'Home', 'score': 150},
  {'category': 'Clothing', 'score': 124},
  {'category': 'Books', 'score': 113}],
 'campaign_type': 'Standard_Reminder'}

In [8]:
client.close()
spark.stop()